In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_annotated/checkpoints/pre_cell_0.pickle

trying: ['dirname']
me:  0
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['Path']
me:  0
trying: ['benchmark_name']
me:  0
trying: ['re']
me:  0
trying: ['filename']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['filenames']
me:  0
Declaring variable dirname
Declaring variable np
Declaring variable pd
Declaring variable Path
Declaring variable benchmark_name
Declaring variable re
Declaring variable filename
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable filenames


In [4]:
%%RecordEvent
%%time
### cell 0 ###

### cell 0 – optimized for cudf
from pathlib import Path
import os, re

base_input = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"
mobile_list = []
broadband_list = []

# precompile regex to extract year and quarter
pattern = re.compile(r"year_(\d+)_quarter_(\d+)\.csv")
for filepath in base_input.rglob("*.csv"):
    # read and convert to nullable dtypes on GPU
    df = pd.read_csv(filepath, thousands=',').convert_dtypes()
    # rename columns if needed
    df.rename(columns={'Number of Record': 'Number of Records'}, inplace=True)
    # extract year and quarter
    yr, qt = map(int, pattern.search(filepath.name).groups())
    df['Year'] = np.int64(yr)
    df['Quarter'] = np.int64(qt)
    # collect per category
    if 'mobile' in filepath.name:
        mobile_list.append(df)
    else:
        broadband_list.append(df)

# single concat per category on GPU
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

# ensure Year/Quarter are int64 and sort
Mobile_df = (Mobile_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)
Broadband_df = (Broadband_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)

# expand datasets
factor = 10
Mobile_df = pd.concat([Mobile_df] * (factor * 2), ignore_index=True)
Broadband_df = pd.concat([Broadband_df] * factor, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df.info()
Broadband_df.info()

(25970, 13)
(49740, 13)
<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 49740 entries, 0 to 49739
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               49740 non-null  string
 1   Number of Records  49740 non-null  Int64
 2   Devices            49740 non-null  Int64
 3   Tests              49740 non-null  Int64
 4   Avg. Avg U Kbps    49740 non-null  Int64
 5   Avg. Avg D Kbps    49740 non-null  Int64
 6   Avg Lat Ms         49740 non-null  Int64
 7   Avg. Pop2005       49740 non-null  Int64
 8   Rank Upload        49740 non-null  Int64
 9   Rank Download      49740 non-null  Int64
 10  Rank Latency       49740 non-null  Int64
 11  Year               49740 non-null  int64
 12  Quarter            49740 non-null  int64
dtypes: Int64(10), int64(2), string(1)
memory usage: 5.2 MB
<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 25970 entries, 0 to 25969
Data columns (total 13 columns):
 # 

In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_0_try_4.pickle

migration speed (bps): 859801643.5087187
---------------------------
variables to migrate:
yr 28
Mobile_df 5474844
Broadband_df 2866634
re 72
filenames 248
df 24651
BENCHMARKS_TO_PATHS 2272
qt 28
filename 80
base_input 88
benchmark_name 63
pattern 440
mobile_list 273970
factor 28
Path 904
broadband_list 286891
filepath 88
dirname 141
np 72
pd 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_0_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['Path', 'benchmark_name', 'BENCHMARKS_TO_PATHS']
Active variables []
Intermediate variables ['Mobile_df', 'broadband_list', 'factor', 'Broadband_df', 'mobile_list', 'filepath', 'yr', 'base_input', 'pattern', 'qt', 'df']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/opt_cell_exec_info_0_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[0], f)


In [8]:
opt_output = Out.get(4)

In [9]:
Mobile_df_opt = Mobile_df
Broadband_df_opt = Broadband_df
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_0.pickle
assert compare_df(Mobile_df_opt, Mobile_df)
assert compare_df(Broadband_df_opt, Broadband_df)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['filenames']
me:  1
trying: ['Path']
me:  0
trying: ['data']
me:  1
trying: ['re']
me:  0
trying: ['dirname']
me:  0
trying: ['Mobile_df']
me:  1
trying: ['factor']
me:  1
trying: ['pd']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['Broadband_df']
me:  1
trying: ['orig_output']
me:  2
trying: ['benchmark_name']
me:  0
trying: ['np']
me:  0
trying: ['col_name_corrections']
me:  1
Declaring variable Path
Declaring variable re
Declaring variable dirname
Declaring variable pd
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable benchmark_name
Declaring variable np
Declaring variable filename
Declaring variable meta_info
Declaring variable filename
Declaring variable meta_info
Declaring variable filenames
Declaring variable data
Declaring variable Mobile_df
Declaring variable factor
Declaring variable Broadband_df
Declaring variable col_name_corrections
Declaring variable 